# Laboratorio 4. Datos Geoespaciales

- Abby Donis 22440
- Hansel Lopez 19026
- Fabian Prado Dluzniewski 23427

Este notebook cubre los incisos 1-2

### 1. Conexion

In [4]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
import openeo
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

In [3]:
conn = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=IYQJ-DJIX 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


In [ ]:
#areas interes
lago_atitlan={
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}

lago_amatitlan ={
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799
}

#carga bandas

fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", 
    "2025-07-17", "2025-11-21", "2025-12-29",
    "2026-02-12", "2026-03-24", "2026-04-13",
    "2026-04-28", "2026-07-22"
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28",
    "2025-11-24", "2026-01-08", "2026-02-02",
    "2026-02-07", "2026-03-29", "2026-04-13",
    "2026-04-28", "2026-06-19"
]

def procesar_fecha(coords, fecha, nom_lago):
    print(f"--- Lago {nom_lago} ({fecha}) ---")
    s2 = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=coords,
        temporal_extent=[fecha, fecha],
        bands=["B02", "B03", "B04", "B08", "B11", "SCL"]
    )

    #bandas individuales
    blue = s2.band("B02")    #Azul (490nm)
    green = s2.band("B03")   #Verde (560nm)
    red = s2.band("B04")     #Rojo (665nm)
    nir = s2.band("B08")     #Infrarrojo cercano (842nm)
    red_edge = s2.band("B05")  # Red Edge (705nm) bacteria

    #NDVI = NIR-RED/NIR+RED
    ndvi = (nir-red)/(nir+red+1e-10)
    ndvi = ndvi.rename("NDVI")
    #NDWI = GREEN-NIR/GREEN+NIR
    ndwi = (green-nir)/(green+nir+1e-10)
    ndwi = ndwi.rename("NDWI")

    #indice bacterias
    #NDCI (Normalised Difference Chlorophyll Index)(GREEN - RED) / (NIR - RED) * BLUE / GREEN
    bacteria = ((green-red)/(nir-red+1e-10))*(blue/(green+1e-10))
    bacteria = bacteria.rename("Cianobacteria")

    #nubes slc
    # 4=vegetación, 5=no vegetado, 6=agua, 7=nubes bajas, 8=nubes medias, 9=nubes altas, 10=cirros, 11=sombra de nubes
    cloud_mask = s2.band("SCL").apply(
        lambda x: (x != 8)&(x != 9)&(x != 10)&(x != 11)
    )
    #aplicar mascara nueves
    ndvi_mask = ndvi.mask(cloud_mask)
    ndwi_mask = ndwi.mask(cloud_mask)
    bacteria_mask = bacteria.mask(cloud_mask)

    #indices comn
    indices = ndvi_mask.merge(ndwi_mask).merge(bacteria_mask)

    return indices.save_result(
        format="NetCDF",
        options={
            "file_path": f"{nom_lago}_{fecha}_indices_completos.nc"
        }
    )

def procesar_fechas(coordenadas, fechas, nombre_lago):
    print(f"\nProcesamiento Lago {nombre_lago}")
    print(f"Total de fechas: {len(fechas)}")
    
    jobs = []
    
    for fecha in fechas:
        try:
            job = procesar_fecha(coordenadas, fecha, nombre_lago)
            jobs.append((fecha, job))
            print(f"Job para {fecha}")
        except Exception as e:
            print(f"Error job para {fecha}: {e}")
    
    return jobs



print("=== DATOS RASTER ===")

#atitlan
jobs_atitlan = procesar_fechas(lago_atitlan, fechas_atitlan, "Atitlan")
#amatitlan
jobs_amatitlan = procesar_fechas(lago_amatitlan, fechas_amatitlan, "Amatitlan")

def descarga_datos(jobs, ):
    print(f"--- Descargando datos {nom_lago}---")

    for fecha, job in jobs:
        job.start_and_wait()
        job.download_resutls()
        print(f"{fecha} finalizado")

descarga_datos(jobs_atitlan, "atitlan")
descarga_datos(jobs_amatitlan, "amatitlan")

print("Procesamiento finalizado")


=== DATOS RASTER ===

Procesamiento Lago Atitlan
Total de fechas: 11
--- Lago Atitlan (2025-01-18) ---
Error job para 2025-01-18: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2025-04-13) ---
Error job para 2025-04-13: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2025-05-13) ---
Error job para 2025-05-13: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2025-07-17) ---
Error job para 2025-07-17: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2025-11-21) ---
Error job para 2025-11-21: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2025-12-29) ---
Error job para 2025-12-29: Invalid band name/index 'B05'. Valid names: ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL']
--- Lago Atitlan (2026-02-12) ---
Error job

ValueError: not enough values to unpack (expected 2, got 1)